# 04: インタラクティブ化合物選択 / Interactive Compound Selection

mols2grid を使って化合物をインタラクティブに絞り込み、購入・合成候補を選択します。

This notebook uses mols2grid for interactive compound browsing and selection.

In [ ]:
from pathlib import Path
import pandas as pd
from rdkit import Chem

from docking_analysis.analysis.properties import add_properties_to_df
from docking_analysis.clustering.chemical import (
    cluster_by_kmeans,
    cluster_by_butina,
    compute_fp_matrix,
    find_optimal_n_clusters,
    plot_tsne,
)
from docking_analysis.visualization.molsgrid import make_selection_grid, export_selection

from docking_analysis.visualization.pymol import (
    write_ranked_poses_pml,
    write_reference_overlay_pml,
)

## 設定 / Configuration

入力ファイルと出力ディレクトリを指定してください。

Specify the input file and output directory.

In [ ]:
# === ユーザー設定 / User Configuration ===
INPUT_CSV = "../results/filtered_compounds.csv"  # 03_selection の出力 / output from 03
INPUT_SDF = "../results/filtered_compounds.sdf"  # SDF with 3D structures (optional)
OUTPUT_DIR = Path("../results/selected")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SMILES_COL = "SMILES"  # SMILES 列名 / SMILES column name
SCORE_COL = "docking_score"  # スコア列名 / score column name

## データ読み込み / Load Data

In [ ]:
df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df)} compounds with columns: {list(df.columns)}")

# SDF から Mol オブジェクトを読み込む（物性計算用）
# Load Mol objects from SDF (for property calculation)
mols = None
if Path(INPUT_SDF).exists():
    supplier = Chem.SDMolSupplier(str(INPUT_SDF), removeHs=False)
    mols = [m for m in supplier if m is not None]
    print(f"Loaded {len(mols)} molecules from SDF")

## 物性計算（未計算の場合） / Property Calculation (if needed)

MW, LogP, HBD, HBA, TPSA, RotBonds, LE, LELP, RO5, Veber がまだ計算されていない場合に自動計算します。

Auto-calculates ligand properties if not already present in the DataFrame.

In [ ]:
if "mw" not in df.columns and mols is not None:
    print("Computing ligand properties...")
    df = add_properties_to_df(df, mols, score_column=SCORE_COL)
    print(f"Added property columns: mw, logp, hbd, hba, tpsa, rot_bonds, le, lelp, ro5_pass, veber_pass")
elif "mw" not in df.columns:
    print("WARNING: No SDF and no property columns. Skipping property calculation.")

## クラスター数の検討 / Cluster Count Analysis

Elbow 法と Silhouette 分析でクラスター数を検討します。

Evaluate optimal cluster count using Elbow and Silhouette methods.

In [ ]:
if mols is not None and SMILES_COL in df.columns:
    # SMILES から Mol を生成（SDF がない場合）
    if mols is None:
        mols = [Chem.MolFromSmiles(s) for s in df[SMILES_COL] if s]

    fp_matrix = compute_fp_matrix(mols, fingerprint="morgan")
    
    print("Elbow analysis:")
    fig_elbow = find_optimal_n_clusters(fp_matrix, method="elbow")
    fig_elbow
else:
    print("Skipping cluster analysis (no molecules available)")

In [ ]:
if mols is not None:
    print("Silhouette analysis:")
    fig_sil = find_optimal_n_clusters(fp_matrix, method="silhouette")
    fig_sil

## 化学的クラスタリング / Chemical Clustering

上のプロットを参考にクラスター数を決定し、クラスタリングを実行します。

Set the cluster count based on the plots above and run clustering.

In [ ]:
N_CLUSTERS = 10  # ← 上のプロットを見て調整 / adjust based on plots above

if mols is not None:
    result = cluster_by_kmeans(mols, n_clusters=N_CLUSTERS, fingerprint="morgan")
    df["km_cluster_id"] = result.cluster_ids
    print(f"Assigned {result.n_clusters} clusters")
    
    # t-SNE 可視化 / t-SNE visualization
    fig_tsne = plot_tsne(result)
    fig_tsne

## インタラクティブ選択 / Interactive Selection

mols2grid でスライダー付きの化合物グリッドを表示します。
化合物をクリックして選択してください。

Browse compounds with interactive sliders. Click to select.

In [ ]:
grid = make_selection_grid(
    df,
    smiles_col=SMILES_COL,
    sort_by=SCORE_COL,
)
grid.display()

## 選択結果のエクスポート / Export Selection

上で選択した化合物を CSV と SDF に出力します。

Export the selected compounds to CSV and SDF files.

In [ ]:
selected = export_selection(
    grid,
    output_csv=OUTPUT_DIR / "selected_compounds.csv",
    output_sdf=OUTPUT_DIR / "selected_compounds.sdf",
    smiles_col=SMILES_COL,
)
print(f"Exported {len(selected)} selected compounds")
selected.head()

## PyMOL 可視化スクリプト生成 / Generate PyMOL Visualization Scripts

選択した化合物をランク順に表示する `.pml` スクリプトと、リファレンスとのオーバーレイ用スクリプトを生成します。

Generates `.pml` scripts for viewing ranked poses in PyMOL and for overlaying with a reference ligand.

In [ ]:
RECEPTOR_PDB = "../data/receptor_clean.pdb"  # ← 受容体 PDB パスを編集 / edit receptor path
REFERENCE_SDF = None  # ← リファレンスリガンド SDF（任意） / reference ligand SDF (optional)

pymol_dir = OUTPUT_DIR / "pymol"
pymol_dir.mkdir(parents=True, exist_ok=True)

# ランク順表示スクリプト / Ranked poses PML
ranked_pml = write_ranked_poses_pml(
    poses_sdf=OUTPUT_DIR / "selected_compounds.sdf",
    protein_pdb=RECEPTOR_PDB,
    output_pml=pymol_dir / "ranked_poses.pml",
    rank_col=SCORE_COL,
    max_poses=20,
)
print(f"Ranked poses PML: {ranked_pml}")

# リファレンスオーバーレイスクリプト（SDF がある場合）
# Reference overlay PML (if reference SDF is available)
if REFERENCE_SDF is not None and Path(REFERENCE_SDF).exists():
    overlay_pml = write_reference_overlay_pml(
        selected_sdf=OUTPUT_DIR / "selected_compounds.sdf",
        reference_sdf=REFERENCE_SDF,
        protein_pdb=RECEPTOR_PDB,
        output_pml=pymol_dir / "reference_overlay.pml",
        tier_col="km_cluster_id" if "km_cluster_id" in df.columns else None,
    )
    print(f"Reference overlay PML: {overlay_pml}")
else:
    print("No reference SDF specified — skipping overlay PML.")
    print("Set REFERENCE_SDF to a co-crystal ligand SDF to enable overlay.")

print(f"\nPyMOL scripts saved to: {pymol_dir}")
print("Open in PyMOL: pymol ranked_poses.pml")